# Training a StyleGAN2-ADA Model using PyTorch / ONNX

This notebook trains a latent-to-image generator (random z → image) using a simplified,
single-file StyleGAN2-ADA implementation. It follows the same flow as the pix2pix notebooks in
this repo (prep → train → export), but the architecture is **unconditional**: there is no
input image, only a random 512-dim latent vector.

**What this is not:** a 1:1 port of NVIDIA's official `stylegan2-ada-pytorch`. That uses custom
CUDA kernels and ∼3000 lines of code. This notebook is a pure-PyTorch reimplementation you
can read top-to-bottom. Expect lower fidelity than the paper, but you can modify every line.

**Defaults:** 256×256 resolution, batch 8. Change `image_size` below to 128 for faster
iteration or 512 if you have the GPU memory.

**Output:** ONNX exports of the EMA generator (`generator_epoch_XXX.onnx`), taking a `[N, 512]`
latent and producing a `[N, 3, H, H]` image in the `[-1, 1]` range.

In [ ]:
# Make sure you are connected to a runtime with a GPU
!nvidia-smi -L

In [ ]:
# Install required packages
!pip install -q matplotlib tqdm onnx

In [ ]:
# Import all dependencies
import copy
import glob
import math
import os
import random
import sys
from types import SimpleNamespace

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.onnx
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.transforms.functional as TF
from torchvision.utils import make_grid, save_image

from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
from IPython.display import clear_output

In [ ]:
# Check if GPU is available
gpu_available = torch.cuda.is_available()
print("GPU is", "available" if gpu_available else "NOT AVAILABLE")

In [ ]:
# OPTIONAL: If you don't have a dataset yet, you can download a pre-existing one.
# If you have a dataset already, you can skip this step.
#!curl -o ds.zip https://algorithmicgaze.s3.amazonaws.com/workshops/2025-research-week/prep/2025-10-03-dataset-pose-points.zip
#!mkdir -p datasets/faces
#!unzip -j -o -qq *.zip -d datasets/faces
# Remove macOS metadata cruft
#!rm -rf datasets/faces/._*

In [ ]:
# Some helper functions for creating/checking directories.
def directory_should_exist(*args):
    dir = os.path.join(*args)
    if not os.path.isdir(dir):
        raise Exception("Path '{}' is not a directory.".format(dir))
    return dir

def ensure_directory(*args):
    dir = os.path.join(*args)
    os.makedirs(dir, exist_ok=True)
    return dir

In [ ]:
# Point the script to the correct dataset folder and configure training.
input_dir = directory_should_exist("datasets/faces")
output_dir = ensure_directory("output-stylegan2")
log_file_path = os.path.join(output_dir, "training_log.txt")

# --- Image settings ---
image_size = 256           # must be a power of 2, >= 8
split_paired = True        # True = dataset has left|right paired images; use left (target) half

# --- Model size ---
latent_dim = 512           # z dimension
w_dim = 512                # w (style) dimension
mapping_layers = 8         # depth of the z -> w MLP
base_channels = 32768      # ch(res) = min(max_channels, base_channels // res)
max_channels = 512

# --- Training ---
epochs = 200
batch_size = 8             # MUST be a multiple of 4 (minibatch std group size)
g_lr = 2e-3
d_lr = 2e-3

# --- Regularization (lazy = applied every N steps, loss scaled to compensate) ---
r1_every = 16              # R1 gradient penalty interval
r1_gamma = 10.0
pl_every = 4               # Path length regularization interval
pl_weight = 2.0
style_mixing_prob = 0.9

# --- ADA ---
ada_target = 0.6           # target sign(D(real)).mean() -- higher means more aug
ada_speed_imgs = 500000    # images seen before p can move by ≈1

# --- Logging ---
sample_interval = 500      # iterations between sample images
snapshot_interval = 1      # epochs between .pth + .onnx snapshots

# --- EMA ---
ema_decay = 0.999

In [ ]:
# Dataset: loads each image file. For pix2pix-style paired images (left|right),
# set split_paired=True and we'll keep only the LEFT half (the target, what we want to generate).
# Otherwise the full image is used as a training sample.

class ImageDataset(Dataset):
    def __init__(self, root_dir, image_size=256, split_paired=False, transform=None):
        self.root_dir = root_dir
        self.image_size = image_size
        self.split_paired = split_paired
        self.transform = transform
        self.image_files = [
            f for f in os.listdir(root_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.image_files[idx])
        image = Image.open(img_path)
        if image.mode != "RGB":
            image = image.convert("RGB")

        if self.split_paired:
            w, h = image.size
            image = image.crop((0, 0, w // 2, h))

        # Resize to training resolution (bicubic preserves detail)
        image = TF.resize(image, [self.image_size, self.image_size],
                          interpolation=TF.InterpolationMode.BICUBIC)

        # Basic horizontal-flip augmentation (cheap and free diversity)
        if random.random() > 0.5:
            image = TF.hflip(image)

        if self.transform:
            image = self.transform(image)
        return image

In [ ]:
# Normalize to [-1, 1] so the generator's output (which we'll also squash to [-1, 1]) matches.
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

dataset = ImageDataset(input_dir, image_size=image_size, split_paired=split_paired, transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2, drop_last=True)

print(f"Dataset: {len(dataset)} images at {image_size}x{image_size}")

In [ ]:
# Sanity-check by showing a few samples
def plot_tensor(ax, title, img):
    img = (img + 1) / 2
    img = img.clamp(0, 1).permute(1, 2, 0).cpu().numpy()
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")

real_batch = next(iter(dataloader))
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i in range(4):
    plot_tensor(axes[i], f"Sample {i}", real_batch[i])
plt.show()

In [ ]:
# === Equalized learning rate ===
# Weights are initialized ~ N(0, 1) and rescaled by 1/sqrt(fan_in) at forward time.
# This keeps the effective learning rate uniform across layers of different fan_in,
# so Adam doesn't over-update small layers and under-update big ones.

class EqualizedLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=True, bias_init=0.0, lr_mul=1.0):
        super().__init__()
        # Dividing the init by lr_mul, then multiplying the forward scale by lr_mul,
        # results in an effective learning rate multiplied by lr_mul**2 through Adam.
        self.weight = nn.Parameter(torch.randn(out_features, in_features).div_(lr_mul))
        self.bias = nn.Parameter(torch.full((out_features,), float(bias_init))) if bias else None
        self.scale = (1.0 / math.sqrt(in_features)) * lr_mul
        self.lr_mul = lr_mul

    def forward(self, x):
        w = self.weight * self.scale
        b = self.bias * self.lr_mul if self.bias is not None else None
        return F.linear(x, w, b)


class EqualizedConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_channels, in_channels, kernel_size, kernel_size))
        self.bias = nn.Parameter(torch.zeros(out_channels)) if bias else None
        fan_in = in_channels * kernel_size * kernel_size
        self.scale = 1.0 / math.sqrt(fan_in)
        self.stride = stride
        self.padding = padding

    def forward(self, x):
        return F.conv2d(x, self.weight * self.scale, self.bias,
                        stride=self.stride, padding=self.padding)

In [ ]:
# === Modulated convolution (the core StyleGAN2 layer) ===
# StyleGAN1 applied style through AdaIN. StyleGAN2 replaces that with weight modulation:
#   1. The style vector scales each input channel of the conv weights.
#   2. We then *demodulate* (divide by L2 norm over in-channels/kernel), which replaces
#      the normalization step of AdaIN and removes "droplet" artifacts.
#
# The math doesn't care whether you multiply the weights by style or the inputs by style
# (convolution is linear). We do the latter: scale x by style, run a regular conv, then
# scale the output by the per-sample demodulation factor. This is mathematically identical
# to the "grouped-conv over batched weights" trick but keeps the conv weights a static
# tensor, which ONNX can export.

class ModulatedConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, w_dim, demodulate=True):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.demodulate = demodulate
        self.padding = kernel_size // 2

        self.weight = nn.Parameter(torch.randn(out_channels, in_channels, kernel_size, kernel_size))
        self.bias = nn.Parameter(torch.zeros(out_channels))
        fan_in = in_channels * kernel_size * kernel_size
        self.scale = 1.0 / math.sqrt(fan_in)

        # Affine: w -> per-input-channel style. Bias init=1 so the first pass is ~identity.
        self.style_affine = EqualizedLinear(w_dim, in_channels, bias_init=1.0)

    def forward(self, x, w):
        B, C, _, _ = x.shape
        style = self.style_affine(w)                             # [B, in_channels]

        # Modulate inputs (equivalent to modulating the weights before conv)
        x = x * style.view(B, C, 1, 1)

        # Regular conv with the equalized-LR-scaled weights
        weight = self.scale * self.weight                        # [out, in, k, k]
        out = F.conv2d(x, weight, padding=self.padding)          # [B, out, H, W]

        # Demodulation factor, per-sample and per-output-channel.
        # d[b, o] = 1 / sqrt( sum over (c, ky, kx) of (scale*weight[o,c,ky,kx] * style[b,c])^2 + eps )
        #        = 1 / sqrt( sum_c style[b,c]^2 * (sum_{ky,kx} (scale*weight[o,c,ky,kx])^2) + eps )
        if self.demodulate:
            w_sq = weight.pow(2).sum(dim=[2, 3])                 # [out, in]
            s_sq = style.pow(2)                                  # [B, in]
            d = torch.rsqrt(s_sq @ w_sq.t() + 1e-8)              # [B, out]
            out = out * d.view(B, -1, 1, 1)

        return out + self.bias.view(1, -1, 1, 1)

In [ ]:
# === Noise injection ===
# Per-pixel Gaussian noise added to each feature map, scaled by a learned constant.
# Gives the generator a cheap source of stochastic detail (hair strands, freckles) so it
# doesn't have to burn w-space capacity on them.

class NoiseInjection(nn.Module):
    def __init__(self):
        super().__init__()
        self.scale = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        noise = torch.randn(x.shape[0], 1, x.shape[2], x.shape[3],
                            device=x.device, dtype=x.dtype)
        return x + self.scale * noise

In [ ]:
# === Mapping network z -> w ===
# An 8-layer MLP with lr_mul=0.01 (so it updates ~100x slower than the rest of the model).
# The intermediate w-space is less entangled than z-space and gives smoother interpolations.

class MappingNetwork(nn.Module):
    def __init__(self, latent_dim=512, w_dim=512, num_layers=8, lr_mul=0.01):
        super().__init__()
        layers = []
        d = latent_dim
        for _ in range(num_layers):
            layers.append(EqualizedLinear(d, w_dim, lr_mul=lr_mul))
            layers.append(nn.LeakyReLU(0.2))
            d = w_dim
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        # Pixel norm on z (keep on the unit hypersphere)
        z = z * torch.rsqrt(z.pow(2).mean(dim=1, keepdim=True) + 1e-8)
        return self.net(z)

In [ ]:
# === Synthesis block & ToRGB ===
# Each synthesis block doubles resolution (bilinear upsample + 2 modulated convs).
# ToRGB projects the current feature map to 3 channels at each resolution; these are summed
# with the upsampled lower-resolution ToRGB outputs ("skip generator" architecture from
# StyleGAN2, which is simpler and more stable than progressive growing).

class SynthesisBlock(nn.Module):
    def __init__(self, in_channels, out_channels, w_dim):
        super().__init__()
        self.conv1 = ModulatedConv2d(in_channels, out_channels, 3, w_dim)
        self.conv2 = ModulatedConv2d(out_channels, out_channels, 3, w_dim)
        self.noise1 = NoiseInjection()
        self.noise2 = NoiseInjection()
        self.act = nn.LeakyReLU(0.2)

    def forward(self, x, w1, w2):
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        x = self.act(self.noise1(self.conv1(x, w1)))
        x = self.act(self.noise2(self.conv2(x, w2)))
        return x


class ToRGB(nn.Module):
    def __init__(self, in_channels, w_dim):
        super().__init__()
        # 1x1 modulated conv, no demodulation (matches the reference)
        self.conv = ModulatedConv2d(in_channels, 3, 1, w_dim, demodulate=False)

    def forward(self, x, w, prev_rgb=None):
        rgb = self.conv(x, w)
        if prev_rgb is not None:
            rgb = rgb + F.interpolate(prev_rgb, scale_factor=2,
                                      mode="bilinear", align_corners=False)
        return rgb

In [ ]:
# === Generator ===
# Mapping network produces a single w from z, then broadcasts it to one w per style input
# (two styles per synthesis block for the two convs, plus one per ToRGB).
# Style mixing: with some probability, use a second z' for a random suffix of the w inputs
# -- this regularizes the network to localize style information across layers.

class Generator(nn.Module):
    def __init__(self, image_size=256, latent_dim=512, w_dim=512, mapping_layers=8,
                 base_channels=32768, max_channels=512):
        super().__init__()
        self.image_size = image_size
        self.latent_dim = latent_dim
        self.w_dim = w_dim
        self.log_size = int(math.log2(image_size))

        def ch(res):
            return min(max_channels, base_channels // res)

        # Channels per resolution: index i = resolution 2**(i+2), i.e. channels[0] is 4x4
        self.channels = [ch(2 ** i) for i in range(2, self.log_size + 1)]

        self.mapping = MappingNetwork(latent_dim, w_dim, num_layers=mapping_layers)

        # Learned 4x4 constant input (replaces the input z -- all variation flows through styles)
        self.const = nn.Parameter(torch.randn(1, self.channels[0], 4, 4))

        # 4x4 block: single conv + noise + ToRGB
        self.first_conv = ModulatedConv2d(self.channels[0], self.channels[0], 3, w_dim)
        self.first_noise = NoiseInjection()
        self.first_torgb = ToRGB(self.channels[0], w_dim)
        self.act = nn.LeakyReLU(0.2)

        # 8x8, 16x16, ..., image_size
        self.blocks = nn.ModuleList()
        self.to_rgbs = nn.ModuleList()
        for i in range(1, len(self.channels)):
            self.blocks.append(SynthesisBlock(self.channels[i - 1], self.channels[i], w_dim))
            self.to_rgbs.append(ToRGB(self.channels[i], w_dim))

    @property
    def num_ws(self):
        # First block: 1 conv + 1 ToRGB = 2 styles
        # Each subsequent block: 2 convs + 1 ToRGB = 3 styles
        return 2 + (len(self.channels) - 1) * 3

    def forward(self, z, mix_z=None, return_w=False):
        B = z.shape[0]

        w = self.mapping(z)
        ws = w.unsqueeze(1).expand(-1, self.num_ws, -1)                 # [B, num_ws, w_dim]

        if mix_z is not None:
            w2 = self.mapping(mix_z)
            ws2 = w2.unsqueeze(1).expand(-1, self.num_ws, -1)
            cut = random.randint(1, self.num_ws - 1)
            ws = torch.cat([ws[:, :cut], ws2[:, cut:]], dim=1)

        x = self.const.expand(B, -1, -1, -1)
        wi = 0
        x = self.act(self.first_noise(self.first_conv(x, ws[:, wi]))); wi += 1
        rgb = self.first_torgb(x, ws[:, wi]); wi += 1

        for block, to_rgb in zip(self.blocks, self.to_rgbs):
            x = block(x, ws[:, wi], ws[:, wi + 1]); wi += 2
            rgb = to_rgb(x, ws[:, wi], prev_rgb=rgb); wi += 1

        if return_w:
            return rgb, ws
        return rgb

In [ ]:
# === Discriminator ===
# Residual discriminator: each block halves resolution via a (conv + avgpool) main path and a
# (1x1 + avgpool) skip path, summed and divided by sqrt(2) to preserve activation variance.
# MinibatchStdDev (at 4x4) adds a feature channel summarizing std across a mini-group; this
# is a cheap anti-mode-collapse signal -- the generator can't fake activation variance.

class MinibatchStdDev(nn.Module):
    def __init__(self, group_size=4):
        super().__init__()
        self.group_size = group_size

    def forward(self, x):
        B, C, H, W = x.shape
        G = min(self.group_size, B)
        # Requires B % G == 0 (ensured by drop_last=True and batch_size=8 with G=4)
        y = x.view(G, -1, C, H, W)
        y = y - y.mean(dim=0, keepdim=True)
        y = y.pow(2).mean(dim=0).sqrt()           # [B/G, C, H, W]
        y = y.mean(dim=[1, 2, 3], keepdim=True)    # [B/G, 1, 1, 1]
        y = y.repeat(G, 1, H, W)                   # [B, 1, H, W]
        return torch.cat([x, y], dim=1)


class DiscriminatorBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = EqualizedConv2d(in_channels, in_channels, 3, padding=1)
        self.conv2 = EqualizedConv2d(in_channels, out_channels, 3, padding=1)
        self.skip = EqualizedConv2d(in_channels, out_channels, 1, bias=False)
        self.act = nn.LeakyReLU(0.2)

    def forward(self, x):
        skip = self.skip(F.avg_pool2d(x, 2))
        x = self.act(self.conv1(x))
        x = F.avg_pool2d(x, 2)
        x = self.act(self.conv2(x))
        return (x + skip) / math.sqrt(2)


class Discriminator(nn.Module):
    def __init__(self, image_size=256, base_channels=32768, max_channels=512):
        super().__init__()
        self.log_size = int(math.log2(image_size))

        def ch(res):
            return min(max_channels, base_channels // res)

        # Channels going DOWN: [ch(image_size), ch(image_size/2), ..., ch(4)]
        channels = [ch(2 ** i) for i in range(self.log_size, 1, -1)]

        self.from_rgb = nn.Sequential(
            EqualizedConv2d(3, channels[0], 1),
            nn.LeakyReLU(0.2),
        )

        self.blocks = nn.ModuleList([
            DiscriminatorBlock(channels[i], channels[i + 1])
            for i in range(len(channels) - 1)
        ])

        self.minibatch_std = MinibatchStdDev()
        # +1 channel for the appended minibatch std
        self.final_conv = EqualizedConv2d(channels[-1] + 1, channels[-1], 3, padding=1)
        self.final_linear1 = EqualizedLinear(channels[-1] * 4 * 4, channels[-1])
        self.final_linear2 = EqualizedLinear(channels[-1], 1)
        self.act = nn.LeakyReLU(0.2)

    def forward(self, x):
        x = self.from_rgb(x)
        for block in self.blocks:
            x = block(x)
        x = self.minibatch_std(x)
        x = self.act(self.final_conv(x))
        x = x.flatten(1)
        x = self.act(self.final_linear1(x))
        return self.final_linear2(x)

In [ ]:
# === Adaptive Discriminator Augmentation (ADA) ===
# On small datasets (~1k-10k images), the discriminator quickly memorizes the real set and
# training collapses. ADA applies random augmentations to BOTH real and fake inputs to the
# discriminator, and adapts the augmentation probability p based on how "confident" the
# discriminator is on reals (sign(D(real)).mean() -- close to +1 means overfitting).
#
# This is a simplified subset of NVIDIA's ADA: hflip, rotation, translation, brightness.
# The full version has ~18 augmentations (color, cutout, frequency-domain, etc.).

class AdaAugment(nn.Module):
    def __init__(self, target=0.6, speed_imgs=500000, max_p=0.8):
        super().__init__()
        self.register_buffer("p", torch.tensor(0.0))
        self.target = target
        self.speed_imgs = speed_imgs
        self.max_p = max_p

    def _affine(self, x, theta):
        grid = F.affine_grid(theta, x.shape, align_corners=False)
        return F.grid_sample(x, grid, align_corners=False, padding_mode="reflection")

    def forward(self, x):
        p = float(self.p.item())
        if p <= 0:
            return x
        B = x.shape[0]
        device = x.device

        # Horizontal flip
        flip_mask = (torch.rand(B, device=device) < p).view(B, 1, 1, 1).float()
        x = flip_mask * TF.hflip(x) + (1 - flip_mask) * x

        # Combined small rotation + translation as a single affine transform
        rot_on = (torch.rand(B, device=device) < p).float()
        angle = (torch.rand(B, device=device) * 2 - 1) * math.radians(20) * rot_on
        trans_on = (torch.rand(B, device=device) < p).float()
        tx = (torch.rand(B, device=device) * 2 - 1) * 0.125 * trans_on
        ty = (torch.rand(B, device=device) * 2 - 1) * 0.125 * trans_on

        cos = torch.cos(angle)
        sin = torch.sin(angle)
        theta = torch.zeros(B, 2, 3, device=device)
        theta[:, 0, 0] = cos
        theta[:, 0, 1] = -sin
        theta[:, 0, 2] = tx * 2        # grid_sample's grid is in [-1, 1]
        theta[:, 1, 0] = sin
        theta[:, 1, 1] = cos
        theta[:, 1, 2] = ty * 2
        x = self._affine(x, theta)

        # Brightness shift
        b_on = (torch.rand(B, device=device) < p).view(B, 1, 1, 1).float()
        x = x + torch.randn(B, 1, 1, 1, device=device) * 0.2 * b_on

        return x

    def update(self, d_real, batch_size):
        # rt in [-1, 1]; +1 means D is too sure reals are real (overfitting signal)
        rt = d_real.detach().sign().mean().item()
        adjust = (rt - self.target) * batch_size / self.speed_imgs
        new_p = float(self.p.item()) + adjust
        self.p.fill_(max(0.0, min(self.max_p, new_p)))

In [ ]:
# === Losses and regularization ===

def d_logistic_loss(d_real, d_fake):
    """Non-saturating logistic GAN loss for the discriminator."""
    return F.softplus(-d_real).mean() + F.softplus(d_fake).mean()


def g_nonsaturating_loss(d_fake):
    """Non-saturating loss for the generator."""
    return F.softplus(-d_fake).mean()


def r1_penalty(discriminator, real_imgs):
    """R1: |∇_x D(x)|^2 at real samples. Pulls D toward a zero-gradient (Lipschitz-ish) regime."""
    real_imgs = real_imgs.detach().requires_grad_(True)
    d_real = discriminator(real_imgs)
    grad = torch.autograd.grad(d_real.sum(), real_imgs, create_graph=True)[0]
    return grad.pow(2).reshape(grad.shape[0], -1).sum(dim=1).mean()


def path_length_penalty(fake_imgs, ws, pl_mean, decay=0.01):
    """Encourages |∇_w (image · noise)| to equal a running-average target length,
    so equal-size steps in w produce equal-size steps in image space."""
    B, _, H, W = fake_imgs.shape
    noise = torch.randn_like(fake_imgs) / math.sqrt(H * W)
    grad = torch.autograd.grad((fake_imgs * noise).sum(), ws, create_graph=True)[0]
    path_lengths = grad.pow(2).sum(dim=2).mean(dim=1).sqrt()
    if pl_mean is None:
        new_pl_mean = path_lengths.mean().detach()
    else:
        new_pl_mean = pl_mean.detach() * (1 - decay) + path_lengths.mean().detach() * decay
    penalty = (path_lengths - new_pl_mean).pow(2).mean()
    return penalty, new_pl_mean

In [ ]:
# === Exponential moving average of generator weights ===
# Maintain a shadow generator whose weights lag the training generator. EMA samples are
# noticeably smoother -- this is what you export at the end.

@torch.no_grad()
def update_ema(ema_model, model, decay=0.999):
    for p_ema, p in zip(ema_model.parameters(), model.parameters()):
        p_ema.data.mul_(decay).add_(p.data, alpha=1 - decay)
    for b_ema, b in zip(ema_model.buffers(), model.buffers()):
        b_ema.data.copy_(b.data)

In [ ]:
# === Snapshot save/load + ONNX export ===

def get_latest_snapshot(output_dir):
    snapshots = glob.glob(os.path.join(output_dir, "snapshot_epoch_*.pth"))
    if not snapshots:
        return None
    return max(snapshots, key=os.path.getctime)


def save_snapshot(path, epoch, generator, g_ema, discriminator,
                  g_optimizer, d_optimizer, ada, pl_mean):
    torch.save({
        "epoch": epoch,
        "generator": generator.state_dict(),
        "g_ema": g_ema.state_dict(),
        "discriminator": discriminator.state_dict(),
        "g_optimizer": g_optimizer.state_dict(),
        "d_optimizer": d_optimizer.state_dict(),
        "ada_p": float(ada.p.item()),
        "pl_mean": float(pl_mean.item()) if pl_mean is not None else None,
    }, path)


def load_snapshot(path, generator, g_ema, discriminator, g_optimizer, d_optimizer, ada, device):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    generator.load_state_dict(ckpt["generator"])
    g_ema.load_state_dict(ckpt["g_ema"])
    discriminator.load_state_dict(ckpt["discriminator"])
    g_optimizer.load_state_dict(ckpt["g_optimizer"])
    d_optimizer.load_state_dict(ckpt["d_optimizer"])
    ada.p.fill_(ckpt["ada_p"])
    pl_mean = torch.tensor(ckpt["pl_mean"], device=device) if ckpt["pl_mean"] is not None else None
    return ckpt["epoch"], pl_mean


def export_onnx(generator, path, latent_dim, device):
    was_training = generator.training
    generator.eval()
    dummy_z = torch.randn(1, latent_dim, device=device)
    with torch.no_grad():
        _ = generator(dummy_z)  # warm up
    torch.onnx.export(
        generator,
        (dummy_z,),
        path,
        export_params=True,
        opset_version=17,
        do_constant_folding=True,
        input_names=["z"],
        output_names=["image"],
        dynamic_axes={"z": {0: "batch_size"}, "image": {0: "batch_size"}},
        dynamo=False,
    )
    if was_training:
        generator.train()

In [ ]:
# === Training loop ===

def train(generator, discriminator, dataloader, opts):
    g_optimizer = optim.Adam(generator.parameters(), lr=opts.g_lr, betas=(0.0, 0.99))
    d_optimizer = optim.Adam(discriminator.parameters(), lr=opts.d_lr, betas=(0.0, 0.99))

    # EMA generator: used for sampling + ONNX export
    g_ema = copy.deepcopy(generator).eval()
    for p in g_ema.parameters():
        p.requires_grad_(False)

    ada = AdaAugment(target=opts.ada_target, speed_imgs=opts.ada_speed_imgs).to(device)
    pl_mean = None

    # Fixed latents so the sample images show progress on the same faces over time
    fixed_z = torch.randn(16, opts.latent_dim, device=device)

    start_epoch = 1
    if not opts.restart:
        latest = get_latest_snapshot(opts.output_dir)
        if latest:
            last_epoch, pl_mean = load_snapshot(
                latest, generator, g_ema, discriminator, g_optimizer, d_optimizer, ada, device
            )
            start_epoch = last_epoch + 1
            print(f"Resuming from epoch {last_epoch}")
        else:
            print("No snapshots found. Starting from scratch.")
    else:
        print("Restarting from scratch.")

    step = 0
    for epoch in range(start_epoch, start_epoch + opts.epochs):
        for i, real_imgs in enumerate(tqdm(dataloader, file=sys.stdout)):
            real_imgs = real_imgs.to(device)
            B = real_imgs.shape[0]

            # ---------------- Discriminator ----------------
            for p in discriminator.parameters():
                p.requires_grad_(True)
            for p in generator.parameters():
                p.requires_grad_(False)
            d_optimizer.zero_grad(set_to_none=True)

            z = torch.randn(B, opts.latent_dim, device=device)
            with torch.no_grad():
                fake_imgs = generator(z)

            real_aug = ada(real_imgs)
            fake_aug = ada(fake_imgs)
            d_real = discriminator(real_aug)
            d_fake = discriminator(fake_aug)
            d_loss = d_logistic_loss(d_real, d_fake)
            d_loss.backward()
            ada.update(d_real, B)

            # Lazy R1: every r1_every steps, scale penalty by r1_every to match continuous version
            r1_val = 0.0
            if step % opts.r1_every == 0:
                r1 = r1_penalty(discriminator, real_aug)
                ((opts.r1_gamma / 2) * r1 * opts.r1_every).backward()
                r1_val = float(r1.item())

            d_optimizer.step()

            # ---------------- Generator ----------------
            for p in discriminator.parameters():
                p.requires_grad_(False)
            for p in generator.parameters():
                p.requires_grad_(True)
            g_optimizer.zero_grad(set_to_none=True)

            z = torch.randn(B, opts.latent_dim, device=device)
            mix_z = torch.randn(B, opts.latent_dim, device=device) \
                if random.random() < opts.style_mixing_prob else None
            fake_imgs = generator(z, mix_z=mix_z)
            fake_aug = ada(fake_imgs)
            d_fake_for_g = discriminator(fake_aug)
            g_loss = g_nonsaturating_loss(d_fake_for_g)
            g_loss.backward()

            # Lazy path length regularization (on a smaller batch for memory)
            pl_val = 0.0
            if step % opts.pl_every == 0:
                pl_batch = max(1, B // 2)
                z_pl = torch.randn(pl_batch, opts.latent_dim, device=device)
                fake_pl, ws_pl = generator(z_pl, return_w=True)
                pl_pen, pl_mean = path_length_penalty(fake_pl, ws_pl, pl_mean)
                (opts.pl_weight * pl_pen * opts.pl_every).backward()
                pl_val = float(pl_pen.item())

            g_optimizer.step()
            update_ema(g_ema, generator, decay=opts.ema_decay)
            step += 1

            # ---------------- Logging ----------------
            if i % 10 == 0:
                log_msg = (
                    f"Epoch {epoch} iter {i} | "
                    f"d_loss: {d_loss.item():.3f} | "
                    f"g_loss: {g_loss.item():.3f} | "
                    f"r1: {r1_val:.3f} | "
                    f"pl: {pl_val:.3f} | "
                    f"ada_p: {ada.p.item():.3f}"
                )
                tqdm.write(log_msg)
                with open(opts.log_file_path, "a") as f:
                    f.write(log_msg + "\n")

            if i % opts.sample_interval == 0:
                g_ema.eval()
                with torch.no_grad():
                    samples = g_ema(fixed_z).cpu()
                    grid = make_grid(samples, nrow=4, normalize=True, value_range=(-1, 1))
                    np_img = grid.permute(1, 2, 0).numpy()
                    clear_output(wait=True)
                    print(f"Epoch {epoch} | ada_p: {ada.p.item():.3f}")
                    plt.figure(figsize=(10, 10))
                    plt.imshow(np_img)
                    plt.axis("off")
                    plt.show()
                    save_image(grid, f"{opts.output_dir}/epoch_{epoch}_iter_{i}.jpg")

        if (epoch + 1) % opts.snapshot_interval == 0:
            snapshot_path = f"{opts.output_dir}/snapshot_epoch_{epoch}.pth"
            save_snapshot(snapshot_path, epoch, generator, g_ema, discriminator,
                          g_optimizer, d_optimizer, ada, pl_mean)
            onnx_path = f"{opts.output_dir}/generator_epoch_{epoch}.onnx"
            export_onnx(g_ema, onnx_path, opts.latent_dim, device)
            print(f"Snapshot:  {snapshot_path}")
            print(f"ONNX:      {onnx_path}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

generator = Generator(
    image_size=image_size,
    latent_dim=latent_dim,
    w_dim=w_dim,
    mapping_layers=mapping_layers,
    base_channels=base_channels,
    max_channels=max_channels,
).to(device)

discriminator = Discriminator(
    image_size=image_size,
    base_channels=base_channels,
    max_channels=max_channels,
).to(device)

print(f"Generator params:     {sum(p.numel() for p in generator.parameters()):,}")
print(f"Discriminator params: {sum(p.numel() for p in discriminator.parameters()):,}")

opts = SimpleNamespace(
    output_dir=output_dir,
    log_file_path=log_file_path,
    image_size=image_size,
    latent_dim=latent_dim,
    epochs=epochs,
    g_lr=g_lr,
    d_lr=d_lr,
    r1_every=r1_every,
    r1_gamma=r1_gamma,
    pl_every=pl_every,
    pl_weight=pl_weight,
    style_mixing_prob=style_mixing_prob,
    ada_target=ada_target,
    ada_speed_imgs=ada_speed_imgs,
    ema_decay=ema_decay,
    sample_interval=sample_interval,
    snapshot_interval=snapshot_interval,
    restart=False,
)

train(generator, discriminator, dataloader, opts)